# DCGAN · Fashion-MNIST

**Generated by GenAI Notebook Generator v1.0.0** · TensorFlow 2 / Keras · Google Colab ready  
Generated at **2026-11-05 05:32 UTC** · Seed **42** · 5 epochs · batch size 64

> This notebook is machine-generated. Review every cell before running it in an
> environment that contains credentials or private data.

## Dataset license

| Field | Value |
| --- | --- |
| **Dataset** | Fashion-MNIST |
| **License** | **MIT** |
| **Attribution** | Zalando SE |
| **Commercial use** | Yes |
| **License URL** | https://github.com/zalandoresearch/fashion-mnist/blob/master/LICENSE |
| **Source** | https://github.com/zalandoresearch/fashion-mnist |
| **Citation** | Xiao, Rasul & Vollgraf (2017). Fashion-MNIST: a Novel Image Dataset. |

**License notes:** Permissive license; retain the copyright and permission notice.

License information is injected from the generator's dataset registry for
convenience. It is not legal advice — verify the current upstream terms before
redistributing data or trained artifacts derived from it.

## Safety & validation

- All Python cells were parsed with `ast` and statically checked for syntax errors.
- Imports and calls matching the malicious-code policy are rejected.
- TF1-only patterns (`tf.compat.v1`, `tf.Session`, `tf.placeholder`, standalone
  Keras imports) are rejected.
- Shell magics are allow-listed; this notebook only installs pinned, trusted
  packages.
- Code generation path: **Deterministic template (LLM disabled or unavailable)**.

## How to run

1. Upload this `.ipynb` to [Google Colab](https://colab.research.google.com/).
2. Select **Runtime → Change runtime type → T4 GPU** (recommended).
3. Select **Runtime → Run all**. Training defaults to 5
   epoch(s); increase `EPOCHS` in the setup cell for better samples.


## 1. Environment Setup

Install the two optional runtime dependencies used by this notebook
(`transformers` for BLIP, `pillow` for image export) and import the
TensorFlow 2 / Keras stack. Google Colab already ships a GPU-enabled
TensorFlow and PyTorch runtime.

**Model:** DCGAN · **Dataset:** Fashion-MNIST (28 × 28 × 1)


In [ ]:
# Colab already provides TensorFlow 2.x and PyTorch. Only BLIP's
# runtime dependencies need to be installed.
!pip -q install "transformers>=4.40,<5" "pillow>=10.0.0"


In [ ]:
%matplotlib inline
import math
import os
import random

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from PIL import Image

SEED = 42
BATCH_SIZE = 64
EPOCHS = 5
LATENT_DIM = 128
MAX_IMAGES = 20000
AUTOTUNE = tf.data.AUTOTUNE

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

def to_display(images):
    """Convert [-1, 1] tensors/arrays into uint8 display images."""
    array = images.numpy() if hasattr(images, "numpy") else np.asarray(images)
    array = np.clip((array + 1.0) * 127.5, 0.0, 255.0).astype("uint8")
    if array.ndim == 4 and array.shape[-1] == 1:
        array = array[..., 0]
    return array

def show_images(images, title="", n=16, save_path=None):
    array = to_display(images)
    n = int(min(n, len(array)))
    cols = 4
    rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 2.1, rows * 2.1))
    axes = np.array(axes).reshape(-1)
    for ax in axes:
        ax.axis("off")
    for ax, image in zip(axes, array[:n]):
        if image.ndim == 2:
            ax.imshow(image, cmap="gray")
        else:
            ax.imshow(image)
    if title:
        fig.suptitle(title, fontsize=14)
    plt.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=140, bbox_inches="tight")
    plt.show()
    plt.close(fig)


## 2. Data Loading

70k grayscale clothing images across 10 classes. Images are downloaded through
`tf.keras.datasets` (no manual files), capped at **20,000**
samples for a fast Colab run, scaled to `[-1, 1]`, and batched.


In [ ]:
(x_train, _), (x_test, _) = tf.keras.datasets.fashion_mnist.load_data()
images = np.concatenate([x_train, x_test], axis=0)[:MAX_IMAGES]

images = images.reshape((-1, 28, 28, 1))
images = images.astype("float32")
images = (images - 127.5) / 127.5

IMAGE_SIZE = 28
CHANNELS = images.shape[-1]
print("images:", images.shape, "min/max:", float(images.min()), float(images.max()))

train_ds = (
    tf.data.Dataset.from_tensor_slices(images)
    .shuffle(buffer_size=min(len(images), 10000), seed=SEED)
    .batch(BATCH_SIZE, drop_remainder=True)
    .prefetch(AUTOTUNE)
)
print("batches per epoch:", len(train_ds))

sample_batch = next(iter(train_ds))
show_images(sample_batch, title="Fashion-MNIST training samples", n=16)


## 3. Model Architecture

A canonical DCGAN pair: the generator projects a 128-D latent vector
into a spatial volume and upsamples with strided transposed
convolutions; the discriminator mirrors it with strided convolutions
and outputs a single logit. Batch normalisation stabilises both
networks, and `tanh` output matches the `[-1, 1]` data range.


In [ ]:
def build_generator():
    start = IMAGE_SIZE // 4
    model = tf.keras.Sequential(
        [
            tf.keras.Input(shape=(LATENT_DIM,)),
            tf.keras.layers.Dense(start * start * 256, use_bias=False),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Reshape((start, start, 256)),
            tf.keras.layers.Conv2DTranspose(
                128, (5, 5), strides=(2, 2), padding="same", use_bias=False
            ),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Conv2DTranspose(
                64, (5, 5), strides=(2, 2), padding="same", use_bias=False
            ),
            tf.keras.layers.BatchNormalization(),
            tf.keras.layers.LeakyReLU(negative_slope=0.2),
            tf.keras.layers.Conv2DTranspose(
                CHANNELS,
                (5, 5),
                strides=(1, 1),
                padding="same",
                activation="tanh",
            ),
        ],
        name="dcgan_generator",
    )
    return model


def build_discriminator():
    inputs = tf.keras.Input(shape=(IMAGE_SIZE, IMAGE_SIZE, CHANNELS))
    x = tf.keras.layers.Conv2D(64, (5, 5), strides=(2, 2), padding="same")(inputs)
    x = tf.keras.layers.LeakyReLU(negative_slope=0.2)(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Conv2D(128, (5, 5), strides=(2, 2), padding="same")(x)
    x = tf.keras.layers.LeakyReLU(negative_slope=0.2)(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Flatten()(x)
    outputs = tf.keras.layers.Dense(1)(x)
    return tf.keras.Model(inputs, outputs, name="dcgan_discriminator")


generator = build_generator()
discriminator = build_discriminator()

print("Generator parameters:", generator.count_params())
print("Discriminator parameters:", discriminator.count_params())
generator.summary()


## 4. Training Loop

The discriminator maximises the probability of real images and
minimises it for generated ones; the generator maximises the
discriminator's confidence on its fakes. A custom `@tf.function`
train step keeps the loop fast, and a 4×4 preview is rendered after
every epoch so progress is visible in Colab.


In [ ]:
binary_cross_entropy = tf.keras.losses.BinaryCrossentropy(from_logits=True)


def discriminator_loss(real_output, fake_output):
    real_loss = binary_cross_entropy(tf.ones_like(real_output), real_output)
    fake_loss = binary_cross_entropy(tf.zeros_like(fake_output), fake_output)
    return real_loss + fake_loss


def generator_loss(fake_output):
    return binary_cross_entropy(tf.ones_like(fake_output), fake_output)


generator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)
discriminator_optimizer = tf.keras.optimizers.Adam(2e-4, beta_1=0.5)


@tf.function
def train_step(images):
    noise = tf.random.normal([tf.shape(images)[0], LATENT_DIM])
    with tf.GradientTape() as gen_tape, tf.GradientTape() as disc_tape:
        generated_images = generator(noise, training=True)
        real_output = discriminator(images, training=True)
        fake_output = discriminator(generated_images, training=True)
        gen_loss = generator_loss(fake_output)
        disc_loss = discriminator_loss(real_output, fake_output)

    gen_gradients = gen_tape.gradient(gen_loss, generator.trainable_variables)
    disc_gradients = disc_tape.gradient(disc_loss, discriminator.trainable_variables)
    generator_optimizer.apply_gradients(
        zip(gen_gradients, generator.trainable_variables)
    )
    discriminator_optimizer.apply_gradients(
        zip(disc_gradients, discriminator.trainable_variables)
    )
    return gen_loss, disc_loss


generator_losses = []
discriminator_losses = []

for epoch in range(1, EPOCHS + 1):
    epoch_generator, epoch_discriminator = [], []
    for image_batch in train_ds:
        g_loss, d_loss = train_step(image_batch)
        epoch_generator.append(float(g_loss))
        epoch_discriminator.append(float(d_loss))

    generator_losses.append(float(np.mean(epoch_generator)))
    discriminator_losses.append(float(np.mean(epoch_discriminator)))

    preview = generator(tf.random.normal([16, LATENT_DIM]), training=False)
    show_images(preview, title="Epoch %d/%d" % (epoch, EPOCHS), n=16)
    print(
        "Epoch %d/%d  generator=%.3f  discriminator=%.3f"
        % (
            epoch,
            EPOCHS,
            generator_losses[-1],
            discriminator_losses[-1],
        )
    )

plt.figure(figsize=(8, 4))
plt.plot(generator_losses, label="generator")
plt.plot(discriminator_losses, label="discriminator")
plt.title("DCGAN training losses")
plt.xlabel("epoch")
plt.ylabel("loss")
plt.legend()
plt.grid(alpha=0.2)
plt.show()


## 5. Inference & Evaluation

Sample the latent space and export one image for the BLIP sanity
check. Early training produces noisy textures; after 20–50 epochs
on Fashion-MNIST the samples should start reading as
grayscale clothing.


In [ ]:
def generate_images(n=16, seed=None):
    if seed is not None:
        tf.random.set_seed(seed)
    noise = tf.random.normal([n, LATENT_DIM])
    return generator(noise, training=False)


samples = generate_images(n=16, seed=2024)
show_images(samples, title="DCGAN generated samples", n=16)

sample_array = to_display(samples[:1])[0]
if sample_array.ndim == 2:
    sample_array = np.stack([sample_array] * 3, axis=-1)

IMAGE_FOR_CAPTION_PATH = "/content/dcgan_sample.png"
Image.fromarray(sample_array).resize((256, 256), Image.NEAREST).save(
    IMAGE_FOR_CAPTION_PATH
)
print("Saved BLIP input to", IMAGE_FOR_CAPTION_PATH)


## 6. BLIP Sanity Check

**U3 — vision-language sanity check.** This cell loads
[`Salesforce/blip-image-captioning-base`](https://huggingface.co/Salesforce/blip-image-captioning-base)
with Hugging Face Transformers and captions the image exported by the
inference cell. Early in training the caption is usually generic
("a close up of a ..."); after enough epochs it should reflect the
dataset domain. This is a qualitative signal, not a metric.

> BLIP runs on PyTorch in this cell; the generative model itself stays
> pure TensorFlow 2 / Keras. Colab pre-installs both frameworks.


In [ ]:
import torch
from PIL import Image
from transformers import BlipForConditionalGeneration, BlipProcessor

caption_device = "cuda" if torch.cuda.is_available() else "cpu"

processor = BlipProcessor.from_pretrained(
    "Salesforce/blip-image-captioning-base"
)
blip_model = BlipForConditionalGeneration.from_pretrained(
    "Salesforce/blip-image-captioning-base"
).to(caption_device)
blip_model.eval()

caption_image = Image.open(IMAGE_FOR_CAPTION_PATH).convert("RGB")
inputs = processor(images=caption_image, return_tensors="pt").to(
    caption_device
)
with torch.no_grad():
    output = blip_model.generate(**inputs, max_new_tokens=40, num_beams=3)
caption = processor.decode(output[0], skip_special_tokens=True)

os.makedirs("/content/outputs", exist_ok=True)
with open("/content/outputs/blip_caption.txt", "w", encoding="utf-8") as handle:
    handle.write(caption + "\n")

print("BLIP caption:", caption)
plt.figure(figsize=(4, 4))
plt.imshow(caption_image)
plt.axis("off")
plt.title(f"BLIP: {caption}")
plt.show()


## 7. Save & Export

Persist the trained weights and write the dataset license alongside the
artifacts so the output bundle stays self-documenting.

**License:** MIT · https://github.com/zalandoresearch/fashion-mnist/blob/master/LICENSE


In [ ]:
os.makedirs("/content/outputs", exist_ok=True)

generator.save("/content/outputs/dcgan_generator.keras")
discriminator.save("/content/outputs/dcgan_discriminator.keras")


with open("/content/outputs/DATASET_LICENSE.txt", "w", encoding="utf-8") as handle:
    handle.write('Fashion-MNIST - MIT\nSource: https://github.com/zalandoresearch/fashion-mnist\nAttribution: Zalando SE\nNotes: Permissive license; retain the copyright and permission notice.\n')

print("Saved artifacts to /content/outputs")
print(os.listdir("/content/outputs"))
